# GliZNet — Full Benchmark Evaluation

Evaluates **GliZNet** (DeBERTa-v3-base backbone, checkpoint-850) on the same 13 datasets used in the GLiClass benchmark paper, reporting **macro F1** for direct comparison.

**Datasets**: CR, SST-2, SST-5, IMDb, 20-Newsgroups, Enron Spam, MASSIVE, Banking77, Financial PhraseBank, AG News, Emotion, CAP-SOTU, Rotten Tomatoes

| Model | Avg macro F1 |
|---|---|
| GLiClass-large-v3.0 | 0.7193 |
| GLiClass-base-v3.0 | 0.6764 |
| deberta-v3-large-zeroshot-v2.0 | 0.6821 |
| deberta-v3-base-zeroshot-v2.0 | 0.6559 |
| **GliZNet-deberta-v3-base (ours)** | **?** |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset, ClassLabel
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

from gliznet import ZeroShotClassificationPipeline
from gliznet.tokenizer import GliZNETTokenizer
from gliznet.model import GliZNetForSequenceClassification

DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"
MODEL_PATH = "alexneakameni/gliznet-deberta-v3-base"
BATCH_SIZE = 32

print(f"Device : {DEVICE}")
print(f"Model  : {MODEL_PATH}")

In [ ]:
# Load GliZNet pipeline
pipeline = ZeroShotClassificationPipeline.from_pretrained(
    MODEL_PATH,
    classification_type="multi-class",
    device=DEVICE,
)
print("✓ GliZNet loaded")

# Sanity check
out = pipeline("I absolutely loved this film!", ["positive", "negative", "neutral"])
for ls in out.labels:
    print(f"  {ls.label:<12} {ls.score:.3f}")

## Dataset loading helpers

In [ ]:
def get_test_split(ds):
    """Return the test (or fallback validation/train) split."""
    for key in ("test", "validation", "train"):
        if key in ds:
            return ds[key]
    raise ValueError("No usable split found")


def resolve_labels(split, label_col):
    """Return (class_names: list[str], int_to_str callable).
    Handles ClassLabel features and plain int/string columns.
    Classes are always returned as strings so the tokenizer never receives ints.
    """
    feat = split.features.get(label_col)
    if isinstance(feat, ClassLabel):
        classes = feat.names          # already list[str]
        to_str = lambda i: classes[i]
    else:
        raw_classes = sorted(set(split[label_col]))
        classes = [str(c) for c in raw_classes]
        to_str = lambda x: str(x)
    return classes, to_str


def prepare(ds, text_col="text", label_col="label", classes=None, to_str=None):
    """Return (texts: list[str], classes: list[str], true_labels: list[str])."""
    split = get_test_split(ds)
    if classes is None:
        classes, to_str = resolve_labels(split, label_col)
    raw = split[label_col]
    true_labels = [to_str(l) for l in raw]
    return list(split[text_col]), classes, true_labels

In [ ]:
def predict_batch(texts, classes, batch_size=BATCH_SIZE):
    """Return the top predicted class string for each text."""
    preds = []
    for i in tqdm(range(0, len(texts), batch_size), desc="  inference", leave=False):
        batch_texts = texts[i : i + batch_size]
        outputs = pipeline(batch_texts, classes)
        for out in outputs:
            best = max(out.labels, key=lambda ls: ls.score)
            preds.append(best.label)
    return preds


def evaluate(preds, true_labels):
    return {
        "macro":    f1_score(true_labels, preds, average="macro"),
        "micro":    f1_score(true_labels, preds, average="micro"),
        "weighted": f1_score(true_labels, preds, average="weighted"),
    }

## Run evaluation on all benchmarks

## Dataset inspection — load each one and check columns / label format

In [ ]:
def inspect(ds_id, config=None, text_col=None, label_col=None):
    """Load a dataset and print its splits, columns, feature types and a few label samples."""
    print(f"\n{'─'*60}")
    print(f"  {ds_id}  {'(' + config + ')' if config else ''}")
    print(f"{'─'*60}")
    ds = load_dataset(ds_id, config) if config else load_dataset(ds_id)
    print(f"  Splits   : {list(ds.keys())}")
    split = get_test_split(ds)
    print(f"  Split    : {len(split)} rows")
    print(f"  Columns  : {split.column_names}")
    print(f"  Features : {split.features}")
    if text_col:
        print(f"  Text[0]  : {str(split[text_col][0])[:120]}")
    if label_col:
        sample_labels = split[label_col][:5]
        print(f"  Labels[0:5]: {sample_labels}")
        feat = split.features.get(label_col)
        if isinstance(feat, ClassLabel):
            print(f"  Class names: {feat.names[:10]}{'...' if len(feat.names)>10 else ''}")
        else:
            unique = sorted(set(split[label_col]))
            print(f"  Unique labels ({len(unique)}): {unique[:10]}{'...' if len(unique)>10 else ''}")

In [ ]:
# ── SetFit/CR ──────────────────────────────────────────────────────────────
inspect("SetFit/CR", text_col="text", label_col="label_text")

In [ ]:
# ── SetFit/sst2 ────────────────────────────────────────────────────────────
inspect("SetFit/sst2", text_col="text", label_col="label_text")

In [ ]:
# ── SetFit/sst5 ────────────────────────────────────────────────────────────
inspect("SetFit/sst5", text_col="text", label_col="label_text")

In [ ]:
# ── stanfordnlp/imdb ───────────────────────────────────────────────────────
inspect("stanfordnlp/imdb", text_col="text", label_col="label")

In [ ]:
# ── SetFit/20_newsgroups ───────────────────────────────────────────────────
inspect("SetFit/20_newsgroups", text_col="text", label_col="label_text")

In [ ]:
# ── SetFit/enron_spam ──────────────────────────────────────────────────────
inspect("SetFit/enron_spam", text_col="text", label_col="label_text")

In [ ]:
# ── mteb/amazon_massive_intent (replaces AmazonScience/massive) ────────────
inspect("mteb/amazon_massive_intent", config="en", text_col="text", label_col="label")

In [ ]:
# ── mteb/banking77 (replaces PolyAI/banking77) ─────────────────────────────
inspect("mteb/banking77", text_col="text", label_col="label_text")

In [ ]:
# ── mteb/financial_phrasebank (replaces takala/financial_phrasebank) ────────
# label is ClassLabel(names=[0,1,2]) → map manually: 0=negative, 1=neutral, 2=positive
inspect("mteb/financial_phrasebank", config="sentences_allagree", text_col="text", label_col="label")

In [ ]:
# ── ag_news ────────────────────────────────────────────────────────────────
inspect("ag_news", text_col="text", label_col="label")

In [ ]:
# ── dair-ai/emotion ────────────────────────────────────────────────────────
inspect("dair-ai/emotion", text_col="text", label_col="label")

In [ ]:
# ── MoritzLaurer/cap_sotu ──────────────────────────────────────────────────
inspect("MoritzLaurer/cap_sotu", text_col="text", label_col="labels")

In [ ]:
# ── cornell-movie-review-data/rotten_tomatoes ──────────────────────────────
inspect("cornell-movie-review-data/rotten_tomatoes", text_col="text", label_col="label")

In [ ]:
DATASETS = [
    "SetFit/CR",
    "SetFit/sst2",
    "SetFit/sst5",
    "stanfordnlp/imdb",
    "SetFit/20_newsgroups",
    "SetFit/enron_spam",
    "mteb/amazon_massive_intent",
    "mteb/banking77",
    "mteb/financial_phrasebank",
    "ag_news",
    "dair-ai/emotion",
    "cornell-movie-review-data/rotten_tomatoes",
]

results_all = {}

for dataset_id in DATASETS:
    print(f"\n{'='*60}")
    print(f"  {dataset_id}")
    print(f"{'='*60}")
    try:
        # ── per-dataset loading ──────────────────────────────────────
        if dataset_id == "stanfordnlp/imdb":
            classes = ["negative", "positive"]
            _to = (lambda c: lambda i: c[i])(classes)
            ds = load_dataset(dataset_id)
            texts, _, true_labels = prepare(ds, label_col="label", classes=classes, to_str=_to)

        elif dataset_id == "SetFit/enron_spam":
            ds = load_dataset(dataset_id)
            texts, classes, true_labels = prepare(ds, label_col="label_text")

        elif dataset_id in ("SetFit/CR", "SetFit/sst2", "SetFit/sst5", "SetFit/20_newsgroups"):
            ds = load_dataset(dataset_id)
            texts, classes, true_labels = prepare(ds, label_col="label_text")

        elif dataset_id == "mteb/amazon_massive_intent":
            ds = load_dataset(dataset_id, "en")
            texts, classes, true_labels = prepare(ds, text_col="text", label_col="label")

        elif dataset_id == "mteb/banking77":
            ds = load_dataset(dataset_id)
            texts, classes, true_labels = prepare(ds, text_col="text", label_col="label_text")

        elif dataset_id == "mteb/financial_phrasebank":
            classes = ["negative", "neutral", "positive"]
            _to = (lambda c: lambda i: c[i])(classes)
            ds = load_dataset(dataset_id, "sentences_allagree")
            texts, _, true_labels = prepare(ds, text_col="text", label_col="label", classes=classes, to_str=_to)

        else:
            ds = load_dataset(dataset_id)
            texts, classes, true_labels = prepare(ds)

        print(f"  Samples : {len(texts)}")
        print(f"  Classes : {len(classes)}  → {classes[:6]}{'...' if len(classes) > 6 else ''}")

        preds  = predict_batch(texts, classes)
        scores = evaluate(preds, true_labels)
        results_all[dataset_id] = scores
        print(f"  macro F1 = {scores['macro']:.4f}  |  micro = {scores['micro']:.4f}")

    except Exception as exc:
        import traceback; traceback.print_exc()
        results_all[dataset_id] = {"macro": float("nan"), "micro": float("nan"), "weighted": float("nan")}

print("\n\n✓ All datasets done")

## Results vs GLiClass benchmark

In [ ]:
GLICLASS_BASE = {
    "SetFit/CR":                                  0.9127,
    "SetFit/sst2":                                0.8959,
    "SetFit/sst5":                                0.3376,
    "stanfordnlp/imdb":                           0.9251,
    "SetFit/20_newsgroups":                       0.4759,
    "SetFit/enron_spam":                          0.6760,
    "mteb/amazon_massive_intent":                 0.5040,
    "mteb/banking77":                             0.4698,
    "mteb/financial_phrasebank":                  0.8971,
    "ag_news":                                    0.7279,
    "dair-ai/emotion":                            0.4447,
    "cornell-movie-review-data/rotten_tomatoes":  0.7943,
}

GLICLASS_LARGE = {
    "SetFit/CR":                                  0.9398,
    "SetFit/sst2":                                0.9192,
    "SetFit/sst5":                                0.4606,
    "stanfordnlp/imdb":                           0.9366,
    "SetFit/20_newsgroups":                       0.5958,
    "SetFit/enron_spam":                          0.7584,
    "mteb/amazon_massive_intent":                 0.5649,
    "mteb/banking77":                             0.5574,
    "mteb/financial_phrasebank":                  0.9000,
    "ag_news":                                    0.7181,
    "dair-ai/emotion":                            0.4506,
    "cornell-movie-review-data/rotten_tomatoes":  0.8411,
}

rows = []
for ds_id in DATASETS:
    r = results_all.get(ds_id, {})
    rows.append({
        "dataset":           ds_id.split("/")[-1],
        "GliZNet (ours)":    round(r.get("macro", float("nan")), 4),
        "GLiClass-base-v3":  GLICLASS_BASE.get(ds_id, float("nan")),
        "GLiClass-large-v3": GLICLASS_LARGE.get(ds_id, float("nan")),
    })

df = pd.DataFrame(rows).set_index("dataset")

avg = df.mean(numeric_only=True)
avg.name = "AVERAGE"
df = pd.concat([df, avg.to_frame().T])
df["Δ vs base"] = (df["GliZNet (ours)"] - df["GLiClass-base-v3"]).round(4)

pd.set_option("display.float_format", "{:.4f}".format)
df

In [ ]:
plot_df = df.drop("AVERAGE").dropna()
x = np.arange(len(plot_df))
w = 0.28

fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(x - w, plot_df["GLiClass-base-v3"],  w, label="GLiClass-base-v3",  color="#3498db", alpha=0.85)
ax.bar(x,     plot_df["GLiClass-large-v3"], w, label="GLiClass-large-v3", color="#9b59b6", alpha=0.85)
ax.bar(x + w, plot_df["GliZNet (ours)"],    w, label="GliZNet (ours)",    color="#e74c3c", alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(plot_df.index, rotation=35, ha="right", fontsize=9)
ax.set_ylabel("Macro F1")
ax.set_ylim(0, 1.05)
ax.set_title("Macro F1 — GliZNet vs GLiClass on standard benchmarks")
ax.legend()

gliznet_avg   = df.loc["AVERAGE", "GliZNet (ours)"]
glibase_avg   = df.loc["AVERAGE", "GLiClass-base-v3"]
glilarge_avg  = df.loc["AVERAGE", "GLiClass-large-v3"]

ax.axhline(gliznet_avg,  color="#e74c3c", ls="--", lw=1.2, alpha=0.5)
ax.axhline(glibase_avg,  color="#3498db", ls="--", lw=1.2, alpha=0.5)

plt.tight_layout()
plt.show()

print(f"\nGliZNet avg macro F1   : {gliznet_avg:.4f}")
print(f"GLiClass-base avg      : {glibase_avg:.4f}")
print(f"GLiClass-large avg     : {glilarge_avg:.4f}")
print(f"Δ vs base              : {gliznet_avg - glibase_avg:+.4f}")